In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import RF_Track as rft
from scipy.optimize import curve_fit
from CLEAR_line import *


In [ ]:

def supergaussian1D(x, A, x0, sigma_x, P):
    return A * np.exp(-( (x-x0)**2 /(2*sigma_x**2) )**P)
    
def r90(sig,P):
    return sig * np.sqrt(2) * (-np.log(0.9))**(1/(2*P))

#80% of central beam
def mask80(x):
    cdf = np.cumsum(x, dtype=float)
    cdf /= cdf[-1] # Normalize CDF to 1
    mask = (cdf >= 0.1) & (cdf <= 0.9) # Mask for central 80%
    return x[mask]

def moving_average(x):
    n = int(len(x)/10)
    """Simple moving average with window size n."""
    return np.convolve(x, np.ones(n)/n, mode='same')
            

def flatness(x):
    
    x = moving_average(x) #smoothing
    x = mask80(x)
    return (max(x)-min(x))/ (max(x)+min(x))

def plot_phsp(T,M, n_bins=50,fov=200, title=None):
     
    def scatter_hist(x, y, ax, ax_histx, ax_histy):
        
        ax_histx.tick_params(axis="x", labelbottom=True)
        ax_histy.tick_params(axis="y", labelleft=True)

        # the scatter plot:
        ax.scatter(x, y,s=1,alpha=0.3)
        ax.set_xlim(-fov,fov)
        ax.set_ylim(-fov,fov)

        ax.set_xlabel('X (mm)')  # Set x-axis label for scatter plot
        ax.set_ylabel('Y (mm)')

        slice_width = 1
        phsp_xslice = M[(M[:,2] < slice_width)]
        phsp_xslice = phsp_xslice[(phsp_xslice[:,2] > -slice_width)]
        
        phsp_yslice = M[(M[:,0] < slice_width)]
        phsp_yslice = phsp_yslice[(phsp_yslice[:,0] > -slice_width)]

        
        hist_x, bin_edges_x = np.histogram(phsp_xslice[:,0], bins=n_bins, range=[-fov, fov])
        bin_centers_x = (bin_edges_x[:-1] + bin_edges_x[1:]) / 2


        hist_y, bin_edges_y = np.histogram(phsp_yslice[:,2], bins=n_bins, range=[-fov, fov])
        bin_centers_y = (bin_edges_y[:-1] + bin_edges_y[1:]) / 2

        #fits
        p0=[np.max(hist_x),  np.mean(phsp_xslice[:,0]), np.std(phsp_xslice[:,0]), 4]
        params_x, _ = curve_fit(supergaussian1D, bin_centers_x, hist_x, p0=p0)

        params_y, _ = curve_fit(supergaussian1D, bin_centers_y, hist_y, p0=p0)
        xy_fit_curve = np.linspace(-fov, fov, 500)
       
        sig_x,sig_y, P_x, P_y = params_x[2], params_y[2], params_x[3], params_y[3]
        r90_x, r90_y = r90(sig_x, P_x), r90(sig_y, P_y)
    
        # Plot SuperGaussian fits

        ax_histx.hist(phsp_xslice[:,0], bins=n_bins, range=[-fov, fov], color="b",alpha=0.6,label= ' X-Intensity')
        ax_histx.plot(xy_fit_curve, supergaussian1D(xy_fit_curve, *params_x), 'r-', label=f"SuperGaussian Fit (P={params_x[3]:.2f},r90={r90_x:.2f})")
        
        ax_histx.legend()


        

        ax_histy.hist(phsp_yslice[:,2], bins=n_bins, range=[-fov, fov], color="b",alpha=0.6,label= ' Y-Intensity',orientation="horizontal")
        ax_histy.plot(supergaussian1D(xy_fit_curve, *params_y), xy_fit_curve,  'r-', label=f"SuperGaussian Fit (P={params_y[3]:.2f},r90={r90_y:.2f})")
        ax_histy.legend()

    fig, axs = plt.subplot_mosaic([['histx', '.'],
                                ['scatter', 'histy']],
                                figsize=(10, 8),
                                width_ratios=(4, 1), height_ratios=(1, 4),
                                layout='constrained')
    scatter_hist(M[:,0], M[:,2], axs['scatter'], axs['histx'], axs['histy'])
    fig.suptitle(title)
    plt.show()



In [ ]:

mass = RF_Track.electronmass    # particle mass in MeV/c^2
population = 10 * RF_Track.nC               # number of particles per bunch                         # particle charge in e units
P_ref = 198  
N_particles = int(100000)
charge = -1

quad_length = float(0.3)  # m
drift_length = float(0.2)


In [ ]:

quad_currents = np.array([19.4, 20.9, 2, 30, 33, 0, 57, 61,8,0,0]) # k1 values from OPTIMISE_CLEAR.py
# quad_currents = np.array([19.4, 20.9, 2, 30, 33, 0, 0,0,0,0,0])
start = 'CA.QFD0350' #'CA.ACS0270S_MECH'
end = 'CA.DHJ0840' #'CA.STLINE$END'
CLEAR_lattice = get_beamline("CLEAR_Beamline_Survey.txt",start, end, P_ref, quad_currents)
print(CLEAR_lattice.get_length())
CLEAR_lattice.append(rft.Drift(0.084))


# S1 = rft.Absorber(0.0001,8.897, 13,26.982,2.7, 166)
# # S1 = rft.Absorber(0.0001,'air')
# S1.disable_energy_straggling()
# S1.set_shape ('circular', 1,1  )
# CLEAR_lattice.append(S1)
# CLEAR_lattice.append(rft.Drift(0.532)) #end of s1 to s2

# s2_thickness = [0.688, 0.778, 0.581, 0.386]
# s2_radii = [0.4, 0.8, 1.2, 1.6]
# for i in range(len(s2_thickness)):
#     Slice = rft.Absorber(s2_thickness[i]/1000,31.9, 37, 288.31,1.32,-1)
#     Slice.disable_energy_straggling()
#     Slice.set_shape ('circular',  abs(s2_radii[i])/1000,abs(s2_radii[i])/1000 )
#     CLEAR_lattice.append(Slice)

# CLEAR_lattice.append(rft.Drift(2.024))  #drift s2 to water tank   




In [ ]:
q = CLEAR_lattice.get_quadrupoles()
for i in q:
    print(i.get_strength())

In [ ]:

# x,xp,y,yp = rft.qrandn(N_particles, 4).T
# momentum = np.ones(N_particles)*P_ref # 200 MeV ± 0.5%
# T = np.zeros(N_particles)
# matrix = np.column_stack((x, xp, y, yp, T, momentum)) #transpose to match Bunch6d format
# B0 = rft.Bunch6d(mass, 0.0, charge, matrix)


In [ ]:

Twiss = RF_Track.Bunch6d_twiss()



Twiss.beta_x = 17.7        # m
Twiss.beta_y = 13.9     # m
Twiss.alpha_x = -1.14 
Twiss.alpha_y = 0.359
Twiss.emitt_x = 4.62     # mm.mrad normalised emittance
Twiss.emitt_y = 3.86     # mm.mrad
# Twiss.sigma_t = 10 * RF_Track.ps       # mm/c   or 37 * RF_Track.ps
# Twiss.sigma_pt = 10     # permille
Twiss.mean_xp = 0.0
Twiss.mean_yp = 0.0

B0 = RF_Track.Bunch6d_QR(mass, population, charge, P_ref, Twiss, N_particles)             # reference bunch



In [ ]:

B1 = CLEAR_lattice.track(B0)
T = CLEAR_lattice.get_transport_table(
'%S %beta_x %beta_y %alpha_x %alpha_y %sigma_x %sigma_y %sigma_px %sigma_py')
M = B1.get_phase_space('%x %xp %y %yp %E %z')

In [ ]:
plot_phsp(T,M,120,3)
# plot_phsp(T,M, n_bins=120,fov=4)


In [ ]:
betas = np.linspace(1,20,3)
alphas = np.linspace(-2,2,3)
for i in range(len(betas)):
    for j in range(len(betas)):
        for k in range(len(alphas)):
            for l in range(len(alphas)):
                Twiss.beta_x = betas[i]        # m
                Twiss.beta_y = betas[j]     # m
                Twiss.alpha_x = alphas[k] 
                Twiss.alpha_y = alphas[l]
                Twiss.emitt_x = 5    # mm.mrad normalised emittance
                Twiss.emitt_y = 5     # mm.mrad
                B0 = RF_Track.Bunch6d_QR(mass, population, charge, P_ref, Twiss, N_particles)
                B1 = CLEAR_lattice.track(B0)
                T = CLEAR_lattice.get_transport_table(
                '%S %beta_x %beta_y %alpha_x %alpha_y %sigma_x %sigma_y %sigma_px %sigma_py')
                M = B1.get_phase_space('%x %xp %y %yp %E %z')
                plot_phsp(T,M,120,4,f'beta_x: {betas[i]:.2f}, beta_y: {betas[j]:.2f}, alpha_x: {alphas[k]:.2f}, alpha_y: {alphas[l]:.2f}')
          
